# Figure S1
Survival of 90-day survivors, restarted at each period start.
Pre-pandemic time 0: onset + 90 days, or onset (immortal time bias).

In [ ]:
import os
from enum import Enum
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

In [ ]:
data_path = '../data/excel_4figures.xlsx'
output_dir = '../data/figures'

DAYS_PER_YEAR = 365.25
LANDMARK_DAYS = 90

# same-day deaths
MIN_DURATION_DAYS = 0.1

# (start, end, color); colors from original paper
PERIODS = {
    'Pre-pandemic (2018-2019)': (pd.Timestamp('2018-01-01'), pd.Timestamp('2020-01-01'), '#5a7a8c'),
    'Pandemic (2020-2021)': (pd.Timestamp('2020-01-01'), pd.Timestamp('2022-01-01'), '#b32f1e'),
    'Post-pandemic (2022-2024)': (pd.Timestamp('2022-01-01'), pd.Timestamp('2025-01-01'), '#95BF8F'),
}
PRE_PANDEMIC = 'Pre-pandemic (2018-2019)'

AT_RISK_YEARS = [0, 1, 2, 3]


class PreOrigin(Enum):
    ONSET = 'onset'
    ONSET_PLUS_90D = 'onset_plus_90d'

In [ ]:
survival_df = pd.read_excel(data_path).dropna(how='all')

onset = pd.to_datetime(survival_df['Onset date'].astype(str), format='%Y%m%d')
death = pd.to_datetime(survival_df['5Y Death date'], format='%d/%m/%Y')

# from dates; Mortality_90d inconsistent
survived_90d = ~((death - onset).dt.days <= LANDMARK_DAYS)
survivors_df = pd.DataFrame({'onset': onset, 'death': death})[survived_90d]
len(survivors_df)

In [ ]:
def period_survival(origin, death_date, end):
    death_in_period = death_date < end
    stop = death_date.where(death_in_period, end)
    duration_years = ((stop - origin).dt.days.clip(lower=MIN_DURATION_DAYS)) / DAYS_PER_YEAR
    return pd.DataFrame({'duration_years': duration_years, 'death': death_in_period})


def build_period_cohorts(pre_origin):
    cohorts = {}
    for period, (start, end, _) in PERIODS.items():
        if period == PRE_PANDEMIC:
            origin = survivors_df['onset']
            if pre_origin == PreOrigin.ONSET_PLUS_90D:
                origin = origin + pd.Timedelta(days=LANDMARK_DAYS)
            cohorts[period] = period_survival(origin, survivors_df['death'], end)
            continue

        alive_df = survivors_df[survivors_df['death'].isna() | (survivors_df['death'] >= start)]
        origin = pd.Series(start, index=alive_df.index)
        cohorts[period] = period_survival(origin, alive_df['death'], end)
    return cohorts


def pairwise_logrank(cohorts):
    periods = list(cohorts)
    p_values = {}
    for i, first in enumerate(periods):
        for second in periods[i + 1:]:
            a, b = cohorts[first], cohorts[second]
            result = logrank_test(a['duration_years'], b['duration_years'],
                                  event_observed_A=a['death'], event_observed_B=b['death'])
            p_values[(first, second)] = round(result.p_value, 2)
    return p_values

In [ ]:
AT_RISK_HEADER_Y = -0.2
AT_RISK_ROW_SPACING = 0.055

# axes fraction
AT_RISK_LINE_X = (-0.12, -0.06)


def draw_at_risk_table(ax, fitters):
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(AT_RISK_LINE_X[0], AT_RISK_HEADER_Y, 'Number at risk:', ha='left', transform=ax.transAxes)

    for row, (period, kmf) in enumerate(fitters.items(), start=1):
        y = AT_RISK_HEADER_Y - row * AT_RISK_ROW_SPACING
        _, _, color = PERIODS[period]
        ax.plot(AT_RISK_LINE_X, [y, y], color=color, transform=ax.transAxes, clip_on=False)

        for year in AT_RISK_YEARS:
            n_at_risk = (kmf.durations >= year).sum()
            ax.text(year, y, n_at_risk, ha='center', va='center', transform=trans)


def plot_period_km(cohorts):
    fig, ax = plt.subplots(1, 1, figsize=(7, 5.5))

    fitters = {}
    for period, cohort_df in cohorts.items():
        _, _, color = PERIODS[period]
        kmf = KaplanMeierFitter(label=f'{period} (n = {len(cohort_df)})')
        kmf.fit(cohort_df['duration_years'], event_observed=cohort_df['death'])
        kmf.plot(ax=ax, color=color, ci_show=False)
        fitters[period] = kmf

    # remove upper and right spine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')

    # Set axes
    ax.set_ylim(0, 1)
    ax.set_xlim(0, AT_RISK_YEARS[-1])
    ax.set_xticks(AT_RISK_YEARS)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
    ax.set_yticklabels(['0', '20', '40', '60', '80', '100'])
    ax.yaxis.set_ticks_position('left')
    ax.set_ylabel('Survival rate (%)')
    ax.set_xlabel('Years from start of period')
    ax.grid(False)

    ax.legend(loc='lower right', frameon=False)
    draw_at_risk_table(ax, fitters)

    # room for at-risk table
    plt.subplots_adjust(left=0.15, bottom=0.33)
    return fig

In [ ]:
os.makedirs(output_dir, exist_ok=True)

for pre_origin in PreOrigin:
    cohorts = build_period_cohorts(pre_origin)
    print(pre_origin.value, pairwise_logrank(cohorts))

    fig = plot_period_km(cohorts)
    for extension in ['pdf', 'png']:
        fig.savefig(os.path.join(output_dir, f'figureS1_period_km_{pre_origin.value}.{extension}'),
                    dpi=600, bbox_inches='tight')